# NB21 — Governed Write-Back, Noise, and Quantum-Advantage Benchmarking

**Quantum Brain Research Laboratory — Alejandro Reynoso**


## Purpose

This notebook tests the full recursive architecture. Classical and quantum-conditioned
systems begin with identical graphs, answer the same sequence of questions, and propose
candidate relations. A governance gate evaluates provenance, reliability, contradiction
coverage and duplication before write-back.

The benchmark uses a strict claim ladder:

1. **quantum difference** — the sampled distribution differs from a classical baseline;
2. **quantum usefulness** — the difference improves a task metric under equal budgets;
3. **quantum computational advantage** — improvement survives end-to-end resource
   accounting against the strongest relevant classical method.

This simulator can test the first two experimentally; it cannot establish the third.


In [ ]:
from pathlib import Path
import importlib.util, subprocess, sys

IN_COLAB = Path("/content").exists()
LAB_ROOT = Path("/content/Quantum_Brain_Lab") if IN_COLAB else Path.cwd() / "Quantum_Brain_Lab"
LAB_ROOT.mkdir(parents=True, exist_ok=True)
DEPS = LAB_ROOT / "_deps"

required = {
    "networkx": "networkx",
    "pandas": "pandas",
    "numpy": "numpy",
    "matplotlib": "matplotlib",
}
missing = [pip_name for module, pip_name in required.items()
           if importlib.util.find_spec(module) is None]
if missing:
    DEPS.mkdir(parents=True, exist_ok=True)
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "--target", str(DEPS), *missing]
    )
    sys.path.insert(0, str(DEPS))

print(f"Laboratory root: {LAB_ROOT}")


In [ ]:
from __future__ import annotations

import hashlib
import itertools
import json
import math
import random
import time
import zipfile
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd

SEED = 271828
RNG = random.Random(SEED)
np.random.seed(SEED)

def stable_hash(value: Any) -> str:
    raw = json.dumps(value, sort_keys=True, default=str).encode("utf-8")
    return hashlib.sha256(raw).hexdigest()

def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()

def cosine_words(a: str, b: str) -> float:
    wa = Counter(x.lower().strip(".,:;!?()") for x in a.split())
    wb = Counter(x.lower().strip(".,:;!?()") for x in b.split())
    keys = set(wa) | set(wb)
    if not keys:
        return 0.0
    va = np.array([wa[k] for k in keys], dtype=float)
    vb = np.array([wb[k] for k in keys], dtype=float)
    den = np.linalg.norm(va) * np.linalg.norm(vb)
    return float(va @ vb / den) if den else 0.0

def build_graph() -> nx.MultiDiGraph:
    """Synthetic governed investment-committee vault."""
    g = nx.MultiDiGraph(graph_version="QB-G0", title="Quantum Brain governed vault")
    nodes = {
        "AlphaBank": ("company", "Diversified bank with floating-rate loans, stable deposits, and legacy systems.", 0.00, 0.92),
        "BetaPayments": ("company", "Cloud-native payments platform with rapid growth, thin margins, and rich transaction data.", 0.20, 0.88),
        "ZetaCloud": ("company", "Enterprise cloud provider with recurring revenue and cyber concentration risk.", 0.05, 0.90),
        "GammaRetail": ("company", "Consumer retailer exposed to imported inventory and discretionary demand.", -0.10, 0.87),
        "DeltaLogistics": ("company", "Regional logistics network with pricing power and fuel exposure.", 0.10, 0.86),
        "EtaInsure": ("company", "Insurer benefiting from reinvestment yields while claims rise with inflation.", 0.00, 0.89),
        "RateShock": ("factor", "Policy rates rise sharply, increasing discount rates and funding costs.", -0.70, 0.96),
        "Regulation": ("factor", "Capital, data, conduct, and model-risk requirements tighten.", -0.40, 0.96),
        "CyberShock": ("factor", "A severe cyber event disrupts critical services and customer trust.", -0.80, 0.98),
        "ConsumerSlowdown": ("factor", "Real disposable income and discretionary demand weaken.", -0.70, 0.95),
        "FXShock": ("factor", "The peso depreciates and imported-input costs rise.", -0.60, 0.94),
        "AcquireBeta": ("decision", "AlphaBank considers acquiring BetaPayments.", 0.00, 0.95),
        "ExpandCredit": ("decision", "AlphaBank considers expanding unsecured consumer credit.", 0.00, 0.95),
        "MigrateCloud": ("decision", "The group considers migrating critical workloads to ZetaCloud.", 0.00, 0.95),
        "HedgeFX": ("decision", "GammaRetail considers increasing its foreign-exchange hedge ratio.", 0.00, 0.95),
        "E01": ("evidence", "Payments data can reduce fraud losses and improve cross-selling at AlphaBank.", 0.85, 0.91),
        "E02": ("evidence", "BetaPayments valuation is highly sensitive to higher discount rates.", -0.90, 0.94),
        "E03": ("evidence", "Legacy-control integration creates a material execution risk.", -0.80, 0.93),
        "E04": ("evidence", "Prior acquisitions performed better when product autonomy was preserved.", 0.70, 0.82),
        "E05": ("evidence", "Tighter data regulation increases the fixed cost of payments integration.", -0.75, 0.92),
        "E06": ("evidence", "A bank-payments data estate improves real-time risk detection.", 0.80, 0.89),
        "E07": ("evidence", "Unsecured credit losses rise nonlinearly in consumer slowdowns.", -0.95, 0.97),
        "E08": ("evidence", "Floating-rate assets initially benefit from higher rates.", 0.60, 0.88),
        "E09": ("evidence", "Deposit repricing can later compress the margin benefit.", -0.55, 0.90),
        "E10": ("evidence", "Independent model validation is required before credit expansion.", -0.65, 0.98),
        "E11": ("evidence", "Cloud migration reduces unit costs and improves analytic flexibility.", 0.75, 0.89),
        "E12": ("evidence", "Single-provider concentration can turn a cyber shock into a systemic outage.", -0.95, 0.97),
        "E13": ("evidence", "Workload segmentation contained a prior service disruption.", 0.65, 0.91),
        "E14": ("evidence", "Multi-cloud resilience reduces concentration but raises coordination cost.", 0.25, 0.85),
        "E15": ("evidence", "Layered FX hedges stabilize gross margin.", 0.75, 0.92),
        "E16": ("evidence", "Over-hedging destroys value if currency weakness reverses.", -0.65, 0.88),
        "E17": ("evidence", "FX collateral calls can create a temporary liquidity shock.", -0.55, 0.90),
        "WeakSignalA": ("signal", "A small merchant cohort is moving from cards to account-to-account payments.", 0.45, 0.67),
        "WeakSignalB": ("signal", "New cyber-insurance exclusions may transfer more outage risk to cloud clients.", -0.50, 0.69),
        "OutcomeAutonomy": ("outcome", "Preserved product autonomy accelerated customer migration in a prior deal.", 0.65, 0.94),
        "OutcomeCredit": ("outcome", "A prior downturn generated losses above the linear stress model.", -0.90, 0.96),
        "OutcomeCloud": ("outcome", "Segmentation reduced recovery time during an earlier outage.", 0.70, 0.95),
        "OutcomeHedge": ("outcome", "The hedge protected margin but triggered a collateral call.", 0.10, 0.95),
    }
    for node_id, (kind, text, polarity, reliability) in nodes.items():
        g.add_node(node_id, kind=kind, text=text, polarity=polarity,
                   reliability=reliability, status="authoritative",
                   source=f"source_{1 + len(node_id) % 9:02d}")
    edges = [
        ("AcquireBeta","AlphaBank","concerns"),("AcquireBeta","BetaPayments","concerns"),
        ("ExpandCredit","AlphaBank","concerns"),("MigrateCloud","ZetaCloud","concerns"),
        ("HedgeFX","GammaRetail","concerns"),("AlphaBank","RateShock","exposed_to"),
        ("AlphaBank","Regulation","exposed_to"),("BetaPayments","RateShock","exposed_to"),
        ("BetaPayments","Regulation","exposed_to"),("BetaPayments","ZetaCloud","depends_on"),
        ("ZetaCloud","CyberShock","exposed_to"),("GammaRetail","FXShock","exposed_to"),
        ("GammaRetail","ConsumerSlowdown","exposed_to"),("GammaRetail","DeltaLogistics","depends_on"),
        ("EtaInsure","RateShock","exposed_to"),("EtaInsure","ConsumerSlowdown","exposed_to"),
        ("RateShock","ConsumerSlowdown","causes"),("CyberShock","Regulation","causes"),
        ("FXShock","ConsumerSlowdown","causes"),
        ("E01","AcquireBeta","supports"),("E02","AcquireBeta","contradicts"),
        ("E03","AcquireBeta","contradicts"),("E04","AcquireBeta","supports"),
        ("E05","AcquireBeta","contradicts"),("E06","AcquireBeta","supports"),
        ("WeakSignalA","BetaPayments","informs"),("WeakSignalA","AcquireBeta","supports"),
        ("E04","OutcomeAutonomy","resulted_in"),("OutcomeAutonomy","AcquireBeta","supports"),
        ("E07","ExpandCredit","contradicts"),("E08","ExpandCredit","supports"),
        ("E09","ExpandCredit","contradicts"),("E10","ExpandCredit","contradicts"),
        ("E07","OutcomeCredit","resulted_in"),("OutcomeCredit","ExpandCredit","contradicts"),
        ("E11","MigrateCloud","supports"),("E12","MigrateCloud","contradicts"),
        ("E13","MigrateCloud","supports"),("E14","MigrateCloud","supports"),
        ("WeakSignalB","CyberShock","informs"),("WeakSignalB","MigrateCloud","contradicts"),
        ("E13","OutcomeCloud","resulted_in"),("OutcomeCloud","MigrateCloud","supports"),
        ("E15","HedgeFX","supports"),("E16","HedgeFX","contradicts"),
        ("E17","HedgeFX","contradicts"),("E15","OutcomeHedge","resulted_in"),
        ("OutcomeHedge","HedgeFX","supports"),
        ("E01","E06","corroborates"),("E03","Regulation","informs"),
        ("E05","Regulation","informs"),("E12","CyberShock","informs"),
        ("E17","FXShock","informs"),("E09","RateShock","informs"),
    ]
    for i, (u, v, rel) in enumerate(edges):
        g.add_edge(u, v, rel=rel, weight=0.65 + 0.35 * ((i % 7) / 6))
    return g

QUERIES = [
    {"id":"Q1","text":"Should AlphaBank acquire BetaPayments under higher rates and tighter regulation?",
     "target":"AcquireBeta","seeds":["RateShock","Regulation"],"risk":"high","expected":"caution"},
    {"id":"Q2","text":"Should AlphaBank expand unsecured credit during a consumer slowdown?",
     "target":"ExpandCredit","seeds":["ConsumerSlowdown","RateShock"],"risk":"high","expected":"caution"},
    {"id":"Q3","text":"How should critical workloads migrate to ZetaCloud under cyber risk?",
     "target":"MigrateCloud","seeds":["CyberShock","Regulation"],"risk":"high","expected":"qualified"},
    {"id":"Q4","text":"How should GammaRetail hedge foreign-exchange depreciation risk?",
     "target":"HedgeFX","seeds":["FXShock","ConsumerSlowdown"],"risk":"medium","expected":"qualified"},
]

def simple_graph(g: nx.MultiDiGraph) -> nx.Graph:
    h = nx.Graph()
    h.add_nodes_from(g.nodes(data=True))
    for u, v, d in g.edges(data=True):
        if h.has_edge(u, v):
            h[u][v]["weight"] += float(d.get("weight", 1.0))
        else:
            h.add_edge(u, v, weight=float(d.get("weight", 1.0)), rels={d.get("rel","related")})
    return h

def enumerate_paths(g: nx.MultiDiGraph, query: dict, cutoff: int = 5, cap: int = 80) -> list[list[str]]:
    h = simple_graph(g)
    starts = list(dict.fromkeys(query["seeds"] + [query["target"]]))
    destinations = [n for n, d in h.nodes(data=True)
                    if d.get("kind") in {"evidence","signal","outcome"}]
    paths = []
    for s in starts:
        for t in destinations:
            if s == t:
                continue
            try:
                for p in nx.all_simple_paths(h, s, t, cutoff=cutoff):
                    if query["target"] in p or any(seed in p for seed in query["seeds"]):
                        paths.append(p)
                        if len(paths) >= cap:
                            break
            except nx.NetworkXNoPath:
                pass
            if len(paths) >= cap:
                break
        if len(paths) >= cap:
            break
    unique = []
    seen = set()
    for p in paths:
        key = tuple(p)
        if key not in seen:
            seen.add(key); unique.append(p)
    return unique

def path_features(g: nx.MultiDiGraph, query: dict, path: list[str]) -> dict:
    attrs = [g.nodes[n] for n in path]
    text = " ".join(a.get("text","") for a in attrs)
    evidence = [a for a in attrs if a.get("kind") in {"evidence","outcome","signal"}]
    polarity = float(np.mean([a.get("polarity",0.0) for a in evidence])) if evidence else 0.0
    reliability = float(np.mean([a.get("reliability",0.5) for a in evidence])) if evidence else 0.5
    relevance = cosine_words(query["text"], text)
    kinds = {a.get("kind") for a in attrs}
    novelty = float(sum(a.get("kind") == "signal" for a in attrs) / max(1, len(path)))
    causal = float(any(g.nodes[n].get("kind") == "factor" for n in path)
                   and any(g.nodes[n].get("kind") in {"decision","outcome"} for n in path))
    contradiction = float(polarity < -0.15)
    support = float(polarity > 0.15)
    bridge = float(len(kinds) / 5.0)
    cost = len(path)
    score = (1.7*relevance + 0.9*reliability + 0.45*novelty + 0.40*causal
             + 0.35*bridge + 0.25*contradiction + 0.20*support - 0.06*cost)
    return {
        "relevance": relevance, "reliability": reliability, "novelty": novelty,
        "causal": causal, "contradiction": contradiction, "support": support,
        "bridge": bridge, "cost": cost, "polarity": polarity, "score": score,
    }

def build_catalog(g: nx.MultiDiGraph, queries: list[dict] = QUERIES) -> pd.DataFrame:
    rows = []
    for q in queries:
        for j, path in enumerate(enumerate_paths(g, q)):
            f = path_features(g, q, path)
            rows.append({"query_id":q["id"],"path_id":f"{q['id']}-P{j:03d}",
                         "path":" -> ".join(path),"nodes":path,**f})
    return pd.DataFrame(rows)

def json_graph(g: nx.MultiDiGraph) -> dict:
    return nx.node_link_data(g, edges="edges")

def load_graph(path: Path) -> nx.MultiDiGraph:
    return nx.node_link_graph(json.loads(path.read_text()), edges="edges",
                              directed=True, multigraph=True)

def ensure_baseline() -> tuple[nx.MultiDiGraph, pd.DataFrame]:
    graph_path = LAB_ROOT / "QB_baseline_graph.json"
    catalog_path = LAB_ROOT / "NB17_path_catalog.csv"
    if graph_path.exists() and catalog_path.exists():
        return load_graph(graph_path), pd.read_csv(catalog_path)
    g = build_graph()
    catalog = build_catalog(g)
    graph_path.write_text(json.dumps(json_graph(g), indent=2))
    catalog.assign(nodes=catalog["nodes"].apply(json.dumps)).to_csv(catalog_path, index=False)
    return g, catalog

def normalized_entropy(p: np.ndarray) -> float:
    p = np.asarray(p, dtype=float)
    p = p[p > 1e-15]
    return float(-(p*np.log(p)).sum()/np.log(max(2,len(p))))

def jensen_shannon(p: np.ndarray, q: np.ndarray) -> float:
    p = np.asarray(p,dtype=float); q=np.asarray(q,dtype=float)
    p=p/p.sum(); q=q/q.sum(); m=0.5*(p+q)
    def kl(a,b):
        mask=a>1e-15
        return float(np.sum(a[mask]*np.log(a[mask]/np.maximum(b[mask],1e-15))))
    return math.sqrt(max(0.0,0.5*kl(p,m)+0.5*kl(q,m)))

def portfolio_metrics(df: pd.DataFrame) -> dict:
    if df.empty:
        return {"paths":0,"mean_score":0,"contradiction_share":0,"support_share":0,
                "novelty":0,"node_coverage":0,"polarity_balance":0}
    nodes = set()
    for value in df["path"]:
        nodes.update(str(value).split(" -> "))
    return {
        "paths":len(df), "mean_score":float(df["score"].mean()),
        "contradiction_share":float(df["contradiction"].mean()),
        "support_share":float(df["support"].mean()),
        "novelty":float(df["novelty"].mean()), "node_coverage":len(nodes),
        "polarity_balance":float(1-abs(df["polarity"].mean())),
    }


## 1. Recursive experimental design


In [ ]:
base,catalog=ensure_baseline()
g_classical=base.copy()
g_quantum=base.copy()

def choose_portfolio(frame:pd.DataFrame,mode:str,k:int=4,noise:float=0.0)->pd.DataFrame:
    work=frame.copy()
    if mode=="classical":
        return work.nlargest(k,"score")
    # Quantum-conditioned sampler: amplitude-like weighting plus explicit diversity.
    adjusted=(work.score + .28*work.contradiction + .22*work.novelty
              + .15*work.bridge + np.random.normal(0,noise,len(work)))
    prob=np.exp(3*(adjusted-adjusted.max()).to_numpy()); prob=prob/prob.sum()
    chosen=[]; available=list(range(len(work)))
    first=int(np.random.choice(available,p=prob)); chosen.append(first); available.remove(first)
    while available and len(chosen)<k:
        candidate_weights=[]
        chosen_sets=[set(work.iloc[i].path.split(" -> ")) for i in chosen]
        for i in available:
            s=set(work.iloc[i].path.split(" -> "))
            redundancy=max(len(s&c)/max(1,len(s|c)) for c in chosen_sets)
            candidate_weights.append(max(.001,prob[i]*(1.35-redundancy)))
        candidate_weights=np.array(candidate_weights); candidate_weights/=candidate_weights.sum()
        j=int(np.random.choice(len(available),p=candidate_weights))
        chosen.append(available[j]); available.pop(j)
    return work.iloc[chosen]

def propose_edge(g:nx.MultiDiGraph,q:dict,selected:pd.DataFrame,round_id:int,mode:str)->dict:
    polarity=float(selected.polarity.mean())
    relation="supports" if polarity>.12 else "contradicts" if polarity<-.12 else "qualifies"
    evidence_nodes=sorted(set(itertools.chain.from_iterable(
        p.split(" -> ") for p in selected.path)))
    return {
        "candidate_id":f"{mode}-{round_id:03d}-{q['id']}",
        "source":evidence_nodes[0],"target":q["target"],"relation":relation,
        "mean_reliability":float(selected.reliability.mean()),
        "contradiction_coverage":float(selected.contradiction.mean()),
        "novelty":float(selected.novelty.mean()),
        "evidence_nodes":evidence_nodes,"path_ids":selected.path_id.tolist(),
        "status":"candidate","mode":mode,"round":round_id,"query_id":q["id"]
    }

def governance(candidate:dict,g:nx.MultiDiGraph)->tuple[bool,str]:
    if candidate["mean_reliability"]<.78:return False,"insufficient_reliability"
    if len(candidate["evidence_nodes"])<4:return False,"insufficient_evidence_breadth"
    if candidate["relation"]=="supports" and candidate["contradiction_coverage"]<.10:
        return False,"missing_challenge_evidence"
    duplicate=any(u==candidate["source"] and v==candidate["target"]
                  and d.get("rel")==candidate["relation"]
                  for u,v,d in g.edges(data=True))
    if duplicate:return False,"duplicate_relation"
    return True,"approved"

def write_candidate(g:nx.MultiDiGraph,c:dict):
    g.add_edge(c["source"],c["target"],rel=c["relation"],weight=.55,
               status="human_approved_simulation",candidate_id=c["candidate_id"],
               provenance_hash=stable_hash(c))


In [ ]:
registry=[]
metrics=[]
rounds=48
for r in range(rounds):
    q=QUERIES[r%len(QUERIES)]
    frame=catalog[catalog.query_id==q["id"]].reset_index(drop=True)
    for mode,g in [("classical",g_classical),("quantum_conditioned",g_quantum)]:
        selected=choose_portfolio(frame,mode,k=4,noise=.03)
        c=propose_edge(g,q,selected,r,mode)
        approved,reason=governance(c,g)
        c.update({"approved":approved,"governance_reason":reason})
        if approved:write_candidate(g,c)
        registry.append(c)
        pm=portfolio_metrics(selected)
        metrics.append({"round":r,"query_id":q["id"],"mode":mode,
                        "approved":approved,**pm,
                        "edges_after":g.number_of_edges()})
recursive_metrics=pd.DataFrame(metrics)
candidate_registry=pd.DataFrame([{k:v for k,v in c.items()
                                  if k not in {"evidence_nodes","path_ids"}} for c in registry])
display(recursive_metrics.groupby("mode").agg(
    approval_rate=("approved","mean"),mean_score=("mean_score","mean"),
    contradiction=("contradiction_share","mean"),novelty=("novelty","mean"),
    coverage=("node_coverage","mean"),final_edges=("edges_after","max")
).round(3))


## 2. Graph evolution and divergence


In [ ]:
hc=simple_graph(g_classical); hq=simple_graph(g_quantum)
def topology(h:nx.Graph)->dict:
    deg=np.array([d for _,d in h.degree()],dtype=float)
    return {"nodes":h.number_of_nodes(),"edges":h.number_of_edges(),
            "density":nx.density(h),"clustering":nx.average_clustering(h),
            "degree_hhi":float(((deg/max(1,deg.sum()))**2).sum()),
            "components":nx.number_connected_components(h)}
edge_c={(u,v,d.get("rel")) for u,v,d in g_classical.edges(data=True)}
edge_q={(u,v,d.get("rel")) for u,v,d in g_quantum.edges(data=True)}
jaccard=1-len(edge_c&edge_q)/max(1,len(edge_c|edge_q))
topology_df=pd.DataFrame([
    {"mode":"classical",**topology(hc)},
    {"mode":"quantum_conditioned",**topology(hq)}
])
topology_df["edge_set_distance"]=jaccard
display(topology_df.round(4))


## 3. Noise and end-to-end benchmarking


In [ ]:
bench=[]
for noise in [0,.03,.08,.15,.30]:
    for q in QUERIES:
        frame=catalog[catalog.query_id==q["id"]].reset_index(drop=True)
        for mode in ["classical","quantum_conditioned"]:
            start=time.perf_counter()
            selected=choose_portfolio(frame,mode,k=4,noise=noise if mode!="classical" else 0)
            elapsed=time.perf_counter()-start
            pm=portfolio_metrics(selected)
            utility=(.35*pm["mean_score"]+.18*pm["polarity_balance"]
                     +.16*pm["contradiction_share"]+.14*pm["novelty"]
                     +.02*pm["node_coverage"]-.04*math.log1p(elapsed*1e5))
            bench.append({"noise":noise,"query_id":q["id"],"mode":mode,
                          "runtime_seconds":elapsed,"utility":utility,**pm})
benchmark=pd.DataFrame(bench)
advantage=benchmark.groupby(["noise","mode"]).agg(
    utility=("utility","mean"),runtime=("runtime_seconds","mean"),
    diversity=("node_coverage","mean"),contradiction=("contradiction_share","mean")
).reset_index()
display(advantage.round(4))


In [ ]:
fig,axes=plt.subplots(1,2,figsize=(12,4.5))
for mode,frame in advantage.groupby("mode"):
    axes[0].plot(frame.noise,frame.utility,marker="o",label=mode)
    axes[1].plot(frame.noise,frame.diversity,marker="o",label=mode)
axes[0].set_title("End-to-end usefulness under sampler noise")
axes[0].set_ylabel("Composite utility (declared objective)")
axes[1].set_title("Context diversity under sampler noise")
axes[1].set_ylabel("Distinct nodes in retained paths")
for ax in axes: ax.set_xlabel("Noise"); ax.grid(alpha=.25); ax.legend()
plt.tight_layout(); plt.savefig(LAB_ROOT/"NB21_advantage_benchmark.png",dpi=180)
plt.show()


## 4. Claim ladder and audit bundle


In [ ]:
q0=benchmark[(benchmark.noise==0)&(benchmark["mode"]=="quantum_conditioned")].utility.mean()
c0=benchmark[(benchmark.noise==0)&(benchmark["mode"]=="classical")].utility.mean()
claim_ladder={
    "quantum_difference":{
        "status":"demonstrated_in_simulation",
        "evidence":"The quantum-conditioned sampling rule produces a different retained portfolio distribution."
    },
    "quantum_usefulness":{
        "status":"provisional" if q0>c0 else "not_demonstrated",
        "equal_budget_utility_delta":float(q0-c0),
        "qualification":"Result depends on the declared utility function and synthetic vault."
    },
    "quantum_computational_advantage":{
        "status":"not_claimed",
        "reason":"No physical QPU, asymptotic scaling study, data-loading cost, or strongest-classical-baseline audit."
    }
}
recursive_metrics.to_csv(LAB_ROOT/"NB21_recursive_metrics.csv",index=False)
candidate_registry.to_csv(LAB_ROOT/"NB21_candidate_registry.csv",index=False)
topology_df.to_csv(LAB_ROOT/"NB21_topology.csv",index=False)
benchmark.to_csv(LAB_ROOT/"NB21_benchmark.csv",index=False)
(LAB_ROOT/"NB21_claim_ladder.json").write_text(json.dumps(claim_ladder,indent=2))
(LAB_ROOT/"NB21_classical_graph.json").write_text(json.dumps(json_graph(g_classical),indent=2))
(LAB_ROOT/"NB21_quantum_graph.json").write_text(json.dumps(json_graph(g_quantum),indent=2))
print(json.dumps(claim_ladder,indent=2))


In [ ]:
audit=LAB_ROOT/"NB21_Audit_Bundle"; audit.mkdir(exist_ok=True)
files=[
    "NB21_recursive_metrics.csv","NB21_candidate_registry.csv","NB21_topology.csv",
    "NB21_benchmark.csv","NB21_claim_ladder.json","NB21_classical_graph.json",
    "NB21_quantum_graph.json"
]
hashes={}
for name in files:
    source=LAB_ROOT/name
    target=audit/name
    target.write_bytes(source.read_bytes())
    hashes[name]=hashlib.sha256(source.read_bytes()).hexdigest()
run_manifest={
    "notebook":"NB21","created_utc":utc_now(),"seed":SEED,"rounds":rounds,
    "initial_graph_hash":stable_hash(json_graph(base)),
    "final_classical_graph_hash":stable_hash(json_graph(g_classical)),
    "final_quantum_graph_hash":stable_hash(json_graph(g_quantum)),
    "governance_rule":"reliability_breadth_challenge_duplicate_gate_v1",
    "claim_ladder":claim_ladder,"file_hashes":hashes,
    "authority":"synthetic_research_output_not_authoritative_knowledge"
}
(audit/"run_manifest.json").write_text(json.dumps(run_manifest,indent=2))
zip_path=LAB_ROOT/"NB21_Audit_Bundle.zip"
with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(audit.iterdir()): zf.write(p,arcname=p.name)
print(f"Audit bundle: {zip_path} ({zip_path.stat().st_size:,} bytes)")


## Conclusion

The full chain is now explicit: governed graph → query compilation → candidate paths →
quantum or quantum-inspired sampling → evidence portfolio → multi-agent deliberation →
governance gate → validated write-back. The laboratory demonstrates how to test a
Quantum Brain without confusing quantum vocabulary with scientific evidence.

The current result is a **hybrid research prototype**, not a claim that present quantum
hardware already outperforms classical graph retrieval. Its value is to make that future
claim falsifiable.
